# Handling Missing Data

---

### Table of Contents
1. Introduction and Setup
2. Finding Missing Data
3. Handling Missing Data: Dropping
4. Handling Missing Data: Filling

---

## 1. Introduction and Setup
- Real-world data is often messy and contains missing values.
- Pandas represents missing values as `np.nan` (Not a Number).
- Most calculations and machine learning algorithms cannot handle missing values,
  so they must be identified and handled first.

In [20]:
import os
import pandas as pd
import numpy as np

# --- Load the sample dataset ---
DATA_FOLDER = "pandas_data"
file_path = os.path.join(DATA_FOLDER, "sample_sales_data.csv")

try:
    df_clean = pd.read_csv(file_path)
    # --- For this lesson, we will programmatically create a "messy" DataFrame ---
    df = df_clean.copy()
    # Introduce some np.nan values for demonstration
    df.loc[2, "Price"] = np.nan
    df.loc[4, "Category"] = np.nan
    df.loc[5, "Price"] = np.nan
    df.loc[5, "Quantity"] = np.nan

    print("--- Messy DataFrame with Missing Values (NaN) ---")
    print(df)

except FileNotFoundError:
    print(f"Error: The data file was not found at '{file_path}'")
    print("Please run '04_reading_and_writing_data.py' first to create it.")
    df = pd.DataFrame()  # Create an empty df to avoid further errors

--- Messy DataFrame with Missing Values (NaN) ---
   OrderID   Product     Category   Price  Quantity   OrderDate
0      101    Laptop  Electronics  1200.0       1.0  2025-01-15
1      102     Mouse  Electronics    25.5       2.0  2025-01-15
2      103  Keyboard  Electronics     NaN       1.0  2025-01-16
3      104   Monitor  Electronics   300.0       2.0  2025-01-17
4      105     Mouse          NaN    27.0       3.0  2025-01-18
5      106    Webcam  Accessories     NaN       NaN  2025-01-18



---

## 2. Finding Missing Data
- Pandas provides easy methods to check for the presence of `NaN`.

In [21]:
# --- `.isnull()` or `.isna()` (They perform the same task) ---

# These return a boolean DataFrame of the same shape.
print("Boolean mask from .isnull():\n", df.isnull())

Boolean mask from .isnull():
    OrderID  Product  Category  Price  Quantity  OrderDate
0    False    False     False  False     False      False
1    False    False     False  False     False      False
2    False    False     False   True     False      False
3    False    False     False  False     False      False
4    False    False      True  False     False      False
5    False    False     False   True      True      False


In [22]:
# --- `.isnull().sum()` ---

# This is a very common idiom to count missing values in each column.
print("\nSum of missing values per column:\n", df.isnull().sum())


Sum of missing values per column:
 OrderID      0
Product      0
Category     1
Price        2
Quantity     1
OrderDate    0
dtype: int64


In [23]:
# --- Filtering for rows with missing data ---
# Find all rows where 'Price' is null.
missing_price_rows = df[df["Price"].isnull()]
print("\nRows where 'Price' is missing:\n", missing_price_rows)


Rows where 'Price' is missing:
    OrderID   Product     Category  Price  Quantity   OrderDate
2      103  Keyboard  Electronics    NaN       1.0  2025-01-16
5      106    Webcam  Accessories    NaN       NaN  2025-01-18



---

## 3. Handling Missing Data: Dropping
- The simplest strategy is to drop rows or columns with missing values.
- Be careful, as this can lead to significant data loss.

In [24]:
print("Original shape:", df.shape)

Original shape: (6, 6)


In [25]:
# --- Drop any ROW containing at least one `NaN` (default behavior) ---

df_dropped_rows = df.dropna()
print("\nDataFrame after dropping any row with NaN:\n", df_dropped_rows)
print("Shape after dropping rows:", df_dropped_rows.shape)


DataFrame after dropping any row with NaN:
    OrderID  Product     Category   Price  Quantity   OrderDate
0      101   Laptop  Electronics  1200.0       1.0  2025-01-15
1      102    Mouse  Electronics    25.5       2.0  2025-01-15
3      104  Monitor  Electronics   300.0       2.0  2025-01-17
Shape after dropping rows: (3, 6)


In [26]:
# --- Drop any COLUMN containing at least one `NaN` ---

df_dropped_cols = df.dropna(axis=1)
print("\nDataFrame after dropping any column with NaN:\n", df_dropped_cols)
print("Shape after dropping columns:", df_dropped_cols.shape)


DataFrame after dropping any column with NaN:
    OrderID   Product   OrderDate
0      101    Laptop  2025-01-15
1      102     Mouse  2025-01-15
2      103  Keyboard  2025-01-16
3      104   Monitor  2025-01-17
4      105     Mouse  2025-01-18
5      106    Webcam  2025-01-18
Shape after dropping columns: (6, 3)



---

## 4. Handling Missing Data: Filling
- Filling (or "imputing") missing values is often preferred over dropping them.

In [27]:
# --- Strategy 1: Fill with a scalar value (e.g., 0) ---

df_filled_zero = df.fillna(0)
print("DataFrame after filling all NaNs with 0:\n", df_filled_zero)

DataFrame after filling all NaNs with 0:
    OrderID   Product     Category   Price  Quantity   OrderDate
0      101    Laptop  Electronics  1200.0       1.0  2025-01-15
1      102     Mouse  Electronics    25.5       2.0  2025-01-15
2      103  Keyboard  Electronics     0.0       1.0  2025-01-16
3      104   Monitor  Electronics   300.0       2.0  2025-01-17
4      105     Mouse            0    27.0       3.0  2025-01-18
5      106    Webcam  Accessories     0.0       0.0  2025-01-18


In [28]:
# --- Strategy 2: Fill with a calculated value (e.g., the mean) ---

# This is a very common strategy for numerical columns
df_filled_mean = df.copy()  # Work on a copy
price_mean = df_filled_mean["Price"].mean()
print(f"\nCalculated mean price: {price_mean:.2f}")
df_filled_mean["Price"] = df_filled_mean["Price"].fillna(price_mean)
print("DataFrame after filling missing Prices with the mean:\n", df_filled_mean)


Calculated mean price: 388.12
DataFrame after filling missing Prices with the mean:
    OrderID   Product     Category     Price  Quantity   OrderDate
0      101    Laptop  Electronics  1200.000       1.0  2025-01-15
1      102     Mouse  Electronics    25.500       2.0  2025-01-15
2      103  Keyboard  Electronics   388.125       1.0  2025-01-16
3      104   Monitor  Electronics   300.000       2.0  2025-01-17
4      105     Mouse          NaN    27.000       3.0  2025-01-18
5      106    Webcam  Accessories   388.125       NaN  2025-01-18


*Note: In a real-world scenario, a better approach would have been to use mean values per product, and fill accordingly.*

In [29]:
# --- Strategy 3: Forward Fill and Backward Fill (`.ffill(), .bfill()`) ---

# This propagates the last valid observation forward. Useful for time-series data.
df_ffill = df.copy()
df_ffill["Price"] = df_ffill["Price"].ffill()
print("\nDataFrame after forward-filling 'Price':\n", df_ffill)


DataFrame after forward-filling 'Price':
    OrderID   Product     Category   Price  Quantity   OrderDate
0      101    Laptop  Electronics  1200.0       1.0  2025-01-15
1      102     Mouse  Electronics    25.5       2.0  2025-01-15
2      103  Keyboard  Electronics    25.5       1.0  2025-01-16
3      104   Monitor  Electronics   300.0       2.0  2025-01-17
4      105     Mouse          NaN    27.0       3.0  2025-01-18
5      106    Webcam  Accessories    27.0       NaN  2025-01-18


In [30]:
# --- Strategy 4: Filling different columns with different values ---

fill_values = {"Category": "Unknown", "Quantity": 0}
df_filled_specific = df.fillna(value=fill_values)
print(
    "\nDataFrame after filling specific columns with different values:\n",
    df_filled_specific,
)


DataFrame after filling specific columns with different values:
    OrderID   Product     Category   Price  Quantity   OrderDate
0      101    Laptop  Electronics  1200.0       1.0  2025-01-15
1      102     Mouse  Electronics    25.5       2.0  2025-01-15
2      103  Keyboard  Electronics     NaN       1.0  2025-01-16
3      104   Monitor  Electronics   300.0       2.0  2025-01-17
4      105     Mouse      Unknown    27.0       3.0  2025-01-18
5      106    Webcam  Accessories     NaN       0.0  2025-01-18



---

**Next:** [Data Manipulation and Transformation](./08_data_manipulation_and_transformation.ipynb)